# Notebook 14b — Fix A-2 with Parametric p-values

**Project:** Data-Driven Cognitive Phenotyping in Acquired Brain Injury  
**Author:** Zoltan Kunos | Universitat de Barcelona  

Notebook 14's empirical bootstrap p-value (`proportion <= 0.5`) is quantized at `1/(B+1) ≈ 0.002`, which sits above the smallest Holm threshold (`0.05/45 ≈ 0.0011`). With B=500 no pair could clear Holm, even though the bootstrap distributions are nowhere near 0.5.

This notebook replaces that test with a **parametric one-sided normal approximation**: `z = (mean - 0.5) / std`, `p = 1 - Phi(z)`. p-values can now be arbitrarily small, so Holm is meaningful. It also reruns the bootstrap with B=1000 for a tighter std estimate, and writes the definitive `Backlog_H2_Holm.csv` plus an updated `backlog_results.pkl` (with the misleading H1 silhouette bootstrap entry replaced by a pointer to NB6's correct re-cluster-per-bootstrap design).

## Imports and load existing backlog state

In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import os, time, warnings
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"

with open(RESULTS / "backlog_results.pkl", "rb") as f:
    backlog = pickle.load(f)

with open(RESULTS / "shared_infrastructure.pkl", "rb") as f:
    infra = pickle.load(f)

## Re-bootstrap pairwise ARIs at B = 1000

Doubling B halves the standard error on the bootstrap mean, which tightens the subsequent z-statistic without changing the parametric inference.

In [2]:
rng = np.random.default_rng(42)
label_arrs = {m: np.asarray(v) for m, v in infra["cluster_labels_fixed"].items()}
methods = list(label_arrs.keys())
n = len(next(iter(label_arrs.values())))
M = len(methods)
B = 1000

pair_mean = {}
pair_std = {}
pair_ci = {}

t0 = time.time()
print(f"Re-bootstrapping {B} iters for parametric p-values...", flush=True)
for i in range(M):
    for j in range(i + 1, M):
        vals = []
        for b in range(B):
            idx = rng.integers(0, n, size=n)
            vals.append(adjusted_rand_score(label_arrs[methods[i]][idx],
                                            label_arrs[methods[j]][idx]))
        arr = np.asarray(vals)
        key = f"{methods[i]}-{methods[j]}"
        pair_mean[key] = float(arr.mean())
        pair_std[key] = float(arr.std(ddof=1))
        pair_ci[key] = (float(np.quantile(arr, 0.025)),
                        float(np.quantile(arr, 0.975)))
print(f"Done ({time.time() - t0:.1f}s)", flush=True)

Re-bootstrapping 1000 iters for parametric p-values...


Done (48.2s)


## Parametric one-sided p-values + Holm and Benjamini–Hochberg correction

H₀: ARI ≤ 0.5 (i.e. agreement is at most chance-corrected mediocre).  
H₁: ARI > 0.5 (i.e. agreement is materially above the threshold).  
Test statistic: `z = (boot_mean - 0.5) / boot_std`,  `p_one = 1 - Φ(z)`.

In [3]:
names, means, stds, p_param, ci_los, ci_his = [], [], [], [], [], []
for key, m in pair_mean.items():
    s = pair_std[key]
    z = (m - 0.5) / s if s > 0 else np.inf
    p_one = 1.0 - norm.cdf(z)
    p_one = max(p_one, 1e-300)
    names.append(key)
    means.append(m)
    stds.append(s)
    p_param.append(p_one)
    ci_los.append(pair_ci[key][0])
    ci_his.append(pair_ci[key][1])

names = np.asarray(names); p_param = np.asarray(p_param)
reject_holm, p_holm, _, _ = multipletests(p_param, alpha=0.05, method="holm")
reject_bh, p_bh, _, _ = multipletests(p_param, alpha=0.05, method="fdr_bh")

df = pd.DataFrame({
    "pair": names,
    "boot_mean_ARI": means,
    "boot_std_ARI": stds,
    "boot_ci_lo": ci_los,
    "boot_ci_hi": ci_his,
    "one_sided_p": p_param,
    "holm_adj_p": p_holm,
    "bh_adj_p": p_bh,
    "reject_Holm_at_0.05": reject_holm,
    "reject_BH_at_0.05": reject_bh,
}).sort_values("boot_mean_ARI").reset_index(drop=True)

df.to_csv(RESULTS / "Backlog_H2_Holm.csv", index=False)

n_pass_holm = int(reject_holm.sum())
n_pass_bh = int(reject_bh.sum())
print(f"Holm at alpha=0.05: {n_pass_holm}/45 pairs reject H0 (ARI > 0.5)")
print(f"BH   at alpha=0.05: {n_pass_bh}/45 pairs reject H0 (ARI > 0.5)")

n_ci_gt_half = int((df.boot_ci_lo > 0.5).sum())
print(f"Pairs with 95% bootstrap CI lower bound > 0.5: {n_ci_gt_half}/45")
df.head(10)

Holm at alpha=0.05: 44/45 pairs reject H0 (ARI > 0.5)
BH   at alpha=0.05: 44/45 pairs reject H0 (ARI > 0.5)
Pairs with 95% bootstrap CI lower bound > 0.5: 44/45


,pair,boot_mean_ARI,boot_std_ARI,boot_ci_lo,boot_ci_hi,one_sided_p,holm_adj_p,bh_adj_p,reject_Holm_at_0.05,reject_BH_at_0.05
0,KNN-SoftImpute,0.509207,0.006136,0.497284,0.521385,6.674400e-02,6.674400e-02,6.674400e-02,False,False
1,KNN-PMM,0.544999,0.006125,0.533584,0.556710,1.012523e-13,2.025047e-13,1.035535e-13,True,True
2,KNN-DAE,0.556976,0.006007,0.544046,0.568517,1.000000e-300,4.500000e-299,1.046512e-300,True,True
3,KNN-NMF,0.561246,0.006419,0.548817,0.574000,1.000000e-300,4.500000e-299,1.046512e-300,True,True
4,KNN-MICE,0.563082,0.006187,0.551653,0.575390,1.000000e-300,4.500000e-299,1.046512e-300,True,True
5,KNN-EM,0.563467,0.006276,0.551202,0.575539,1.000000e-300,4.500000e-299,1.046512e-300,True,True
6,KNN-VAE,0.565121,0.006045,0.553494,0.576497,1.000000e-300,4.500000e-299,1.046512e-300,True,True
7,Mean-KNN,0.590964,0.006281,0.578532,0.602650,1.000000e-300,4.500000e-299,1.046512e-300,True,True
8,KNN-MissForest,0.620984,0.005985,0.608747,0.632950,1.000000e-300,4.500000e-299,1.046512e-300,True,True
9,Mean-PMM,0.644337,0.005795,0.633247,0.655959,1.000000e-300,4.500000e-299,1.046512e-300,True,True


## Update `backlog_results.pkl` and fix the H1 silhouette entry

The empirical-p `A2` block is replaced with the parametric Holm/BH summary. The misleading `H1_silhouette` bootstrap from NB14 is removed in favour of the correct re-cluster-per-bootstrap result already reported by NB6.

In [4]:
backlog["A2"] = dict(
    n_pairs=45,
    method="parametric_normal_approx_with_Holm_and_BH",
    threshold_ARI=0.5,
    alpha=0.05,
    n_reject_Holm=n_pass_holm,
    n_reject_BH=n_pass_bh,
    n_ci_lower_above_half=n_ci_gt_half,
    pair_results={
        row["pair"]: dict(
            boot_mean=float(row["boot_mean_ARI"]),
            boot_std=float(row["boot_std_ARI"]),
            ci_lo=float(row["boot_ci_lo"]),
            ci_hi=float(row["boot_ci_hi"]),
            p_one_sided=float(row["one_sided_p"]),
            holm_p=float(row["holm_adj_p"]),
            bh_p=float(row["bh_adj_p"]),
            reject_Holm=bool(row["reject_Holm_at_0.05"]),
            reject_BH=bool(row["reject_BH_at_0.05"]),
        )
        for _, row in df.iterrows()
    },
)

if "H1_silhouette" in backlog["A3"]:
    del backlog["A3"]["H1_silhouette"]
backlog["A3"]["H1_silhouette_note"] = (
    "Omitted here: NB6 already reports bootstrap mean silhouette 0.263 "
    "(sigma=0.102) using the correct re-cluster-per-bootstrap design; the "
    "approx 95% CI is therefore [0.06, 0.47]."
)

with open(RESULTS / "backlog_results.pkl", "wb") as f:
    pickle.dump(backlog, f)

print("Updated results/Backlog_H2_Holm.csv and backlog_results.pkl")

Updated results/Backlog_H2_Holm.csv and backlog_results.pkl
